# Trace Analysis

This notebook demonstrates how to analyse distributed traces exported by the observability stack's DuckDB export CLI.

The export tool writes span data into a DuckDB database file with the following schema:

```sql
CREATE TABLE spans (
    trace_id        VARCHAR NOT NULL,
    span_id         VARCHAR NOT NULL,
    parent_span_id  VARCHAR,
    operation       VARCHAR NOT NULL,
    service_name    VARCHAR NOT NULL,
    span_kind       VARCHAR,
    start_time      TIMESTAMP NOT NULL,
    end_time        TIMESTAMP NOT NULL,
    duration_us     BIGINT NOT NULL,
    status_code     VARCHAR,
    status_message  VARCHAR,
    attributes      MAP(VARCHAR, VARCHAR),
    resource_attrs  MAP(VARCHAR, VARCHAR),
    export_time     TIMESTAMP
);
```

With DuckDB you can run trace analysis queries that would typically require a dedicated
tracing backend -- latency percentiles, error breakdowns, and service dependency graphs --
all from a single file.

In [ ]:
# Install duckdb if not already available
%pip install --quiet duckdb

import duckdb

# Connect to the exported DuckDB file.
# Replace the path below with the actual exported database file.
#
# List available files with:  !ls ../../observability-export/telemetry-*.duckdb

DB_PATH = "../../observability-export/telemetry-export.duckdb"  # <-- update this path

con = duckdb.connect(DB_PATH, read_only=True)
print(f"Connected to {DB_PATH}")
print(f"Tables: {[row[0] for row in con.execute('SHOW TABLES').fetchall()]}")

In [ ]:
# P95 latency by service
#
# Shows the 95th percentile duration for each service alongside
# total span count, giving a high-level view of service performance.

con.sql("""
    SELECT
        service_name,
        COUNT(*)                                                          AS span_count,
        AVG(duration_us)                                                  AS avg_duration_us,
        PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY duration_us)         AS p95_duration_us
    FROM spans
    GROUP BY service_name
    ORDER BY p95_duration_us DESC
""").show()

In [ ]:
# Span count by operation and error rates
#
# Lists the top operations by volume with their error rate.
# The OpenTelemetry convention uses status_code = 'ERROR' for failed spans.

con.sql("""
    SELECT
        service_name,
        operation,
        COUNT(*)                                                AS total_spans,
        COUNT(*) FILTER (WHERE status_code = 'ERROR')           AS error_count,
        ROUND(100.0 * COUNT(*) FILTER (WHERE status_code = 'ERROR') / COUNT(*), 2)
                                                                AS error_rate_pct
    FROM spans
    GROUP BY service_name, operation
    ORDER BY total_spans DESC
    LIMIT 20
""").show()

In [ ]:
# Service dependency graph
#
# Discovers caller-callee relationships between services by joining
# spans on trace_id and parent_span_id. Only cross-service calls are
# included, forming the edges of a service dependency graph.

con.sql("""
    SELECT
        parent.service_name  AS caller,
        child.service_name   AS callee,
        COUNT(*)             AS call_count
    FROM spans child
    JOIN spans parent
        ON  child.trace_id       = parent.trace_id
        AND child.parent_span_id = parent.span_id
    WHERE child.service_name != parent.service_name
    GROUP BY caller, callee
    ORDER BY call_count DESC
""").show()